# Investment Agent — Diagnostics & Data Source Testing

This notebook lets you test **every API** and **every agent** independently.  
Run cells top-to-bottom, or jump to any section you want to debug.

**Sections:**
1. Setup & Config Validation
2. yfinance — OHLCV, Fundamentals, Macro Data
3. Technical Indicators
4. Finnhub — Sentiment, Analysts, Price Targets, Insiders
5. Alpha Vantage — News Sentiment
6. Trading 212 — Account, Portfolio, Instruments
7. Sub-Strategies — Momentum, Mean Reversion, Factor
8. Strategy Engine — Claude Synthesis
9. Moderation Panel — GPT-4o & Gemini Reviews
10. Risk Manager — Hard Rule Checks

---

## 1. Setup & Config Validation

In [ ]:
import sys, os, json
from pathlib import Path
from pprint import pprint

# Ensure project root is on the path
PROJECT_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Load .env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

In [ ]:
# Validate that all required env vars are set (masked)
REQUIRED_KEYS = [
    "T212_API_KEY", "T212_API_SECRET",
    "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_AI_API_KEY",
    "FINNHUB_API_KEY", "ALPHA_VANTAGE_API_KEY",
]

print("Environment variable check:")
print("-" * 40)
all_ok = True
for key in REQUIRED_KEYS:
    val = os.getenv(key)
    if val:
        masked = val[:4] + "..." + val[-4:] if len(val) > 8 else "****"
        print(f"  {key}: {masked}")
    else:
        print(f"  {key}: MISSING")
        all_ok = False

print()
print("All keys present" if all_ok else "Some keys are MISSING — affected cells will fail")

In [ ]:
# Load settings.yaml
from src.utils.config import get_settings

settings = get_settings()
print("Settings loaded successfully")
print(f"  T212 base URL:  {settings.t212_base_url}")
print(f"  Strategy model: {settings.strategy_model}")
print(f"  Moderator 1:    {settings.moderator_1_model}")
print(f"  Moderator 2:    {settings.moderator_2_model}")
print(f"  Max positions:  {settings.max_positions}")
print(f"  Daily budgets:  Anthropic £{settings.anthropic_daily_gbp} | OpenAI £{settings.openai_daily_gbp} | Google £{settings.google_daily_gbp}")
print(f"  Monthly cap:    £{settings.total_monthly_gbp}")

---
## 2. yfinance — OHLCV, Fundamentals, Macro Data

Free, no API key needed. This is the backbone data source.

In [ ]:
# Test stock for all cells
TEST_TICKER = "AAPL"

In [ ]:
# 2a. OHLCV price data
import yfinance as yf
import pandas as pd

df = yf.download(TEST_TICKER, period="1y", interval="1d", progress=False)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f"{TEST_TICKER} — {len(df)} trading days loaded")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Latest close: ${df['Close'].iloc[-1]:.2f}")
print(f"52-week high: ${df['Close'].max():.2f}")
print(f"52-week low:  ${df['Close'].min():.2f}")
print()
df.tail(5)

In [ ]:
# 2b. Fundamentals via yfinance
from src.agents.market_data.fundamentals import get_fundamentals

fundamentals = get_fundamentals(TEST_TICKER)
print(f"Fundamentals for {TEST_TICKER}:")
print("-" * 40)
for k, v in fundamentals.items():
    if isinstance(v, float) and v is not None:
        print(f"  {k:25s}: {v:>12.4f}")
    else:
        print(f"  {k:25s}: {v}")

In [ ]:
# 2c. Macro data (VIX, yield spread, S&P 500 vs 200-day MA)
from src.agents.market_data.data_fetcher import DataFetcher

fetcher = DataFetcher()
macro = fetcher.get_macro_data()

print("Macro Environment:")
print("-" * 40)
print(f"  Market regime:    {macro.get('market_regime')}")
print(f"  VIX:              {macro.get('vix', 'N/A')}")
print(f"  10Y yield:        {macro.get('ten_year_yield', 'N/A')}")
print(f"  Yield spread:     {macro.get('yield_spread_10y_short', 'N/A')}")
print(f"  S&P 500:          {macro.get('sp500_price', 'N/A')}")
print(f"  S&P 200-day MA:   {macro.get('sp500_200ma', 'N/A')}")
print(f"  S&P above 200MA:  {macro.get('sp500_above_200ma', 'N/A')}")
print(f"  % above 200MA:    {macro.get('sp500_pct_above_200ma', 'N/A')}")

---
## 3. Technical Indicators

Calculated locally from yfinance OHLCV data using the `ta` library. No API key needed.

In [ ]:
from src.agents.market_data.indicators import calculate_indicators, calculate_relative_strength

indicators = calculate_indicators(df)

print(f"Technical Indicators for {TEST_TICKER}:")
print("=" * 50)

# Price
print(f"\n  Price")
print(f"  {'Current price':25s}: ${indicators.get('current_price', 0):.2f}")

# RSI
rsi = indicators.get('rsi_14', 0)
rsi_label = "OVERBOUGHT" if rsi > 70 else "OVERSOLD" if rsi < 30 else "NEUTRAL"
print(f"\n  RSI")
print(f"  {'RSI(14)':25s}: {rsi:.1f}  [{rsi_label}]")

# MACD
print(f"\n  MACD")
print(f"  {'MACD line':25s}: {indicators.get('macd_line', 0):.4f}")
print(f"  {'MACD signal':25s}: {indicators.get('macd_signal', 0):.4f}")
print(f"  {'MACD histogram':25s}: {indicators.get('macd_histogram', 0):.4f}")
print(f"  {'Bullish crossover':25s}: {indicators.get('macd_bullish_crossover')}")
print(f"  {'Bearish crossover':25s}: {indicators.get('macd_bearish_crossover')}")

# Bollinger Bands
print(f"\n  Bollinger Bands")
print(f"  {'Upper':25s}: ${indicators.get('bb_upper', 0):.2f}")
print(f"  {'Middle (20-day MA)':25s}: ${indicators.get('bb_middle', 0):.2f}")
print(f"  {'Lower':25s}: ${indicators.get('bb_lower', 0):.2f}")
print(f"  {'%B (position in band)':25s}: {indicators.get('bb_pct', 0):.2f}")
print(f"  {'Below lower band':25s}: {indicators.get('below_lower_bb')}")

# Moving Averages
print(f"\n  Moving Averages")
print(f"  {'20-day MA':25s}: ${indicators.get('ma_20', 0):.2f}")
print(f"  {'50-day MA':25s}: ${indicators.get('ma_50', 0):.2f}")
print(f"  {'200-day MA':25s}: ${indicators.get('ma_200', 0):.2f}")
print(f"  {'Above 50-day MA':25s}: {indicators.get('above_50ma')}")
print(f"  {'Above 200-day MA':25s}: {indicators.get('above_200ma')}")
print(f"  {'Golden cross':25s}: {indicators.get('golden_cross')}")
print(f"  {'Death cross':25s}: {indicators.get('death_cross')}")

# ATR
print(f"\n  Volatility")
print(f"  {'ATR(14)':25s}: ${indicators.get('atr_14', 0):.2f}")

In [ ]:
# Relative strength vs S&P 500
benchmark_df = fetcher.get_benchmark_data()
rs = calculate_relative_strength(df, benchmark_df)

rs_label = "OUTPERFORMING" if rs > 1.0 else "UNDERPERFORMING" if rs < 1.0 else "IN LINE"
print(f"Relative Strength vs S&P 500 (6-month): {rs:.3f}  [{rs_label}]")

---
## 4. Finnhub — Sentiment, Analysts, Price Targets, Insiders

**API key required:** `FINNHUB_API_KEY`  
**Rate limit:** 60 requests/min (free tier)

In [ ]:
from src.agents.market_data.finnhub_client import FinnhubClient

finnhub = FinnhubClient()
print(f"Finnhub client initialised (base URL: {finnhub.base_url})")

In [ ]:
# 4a. News Sentiment
news_sentiment = finnhub.get_news_sentiment(TEST_TICKER)
print(f"News Sentiment for {TEST_TICKER}:")
pprint(news_sentiment)

In [ ]:
# 4b. Analyst Recommendations
analyst_recs = finnhub.get_analyst_recommendations(TEST_TICKER)
print(f"Analyst Recommendations for {TEST_TICKER}:")
pprint(analyst_recs)

In [ ]:
# 4c. Price Targets
price_targets = finnhub.get_price_targets(TEST_TICKER)
print(f"Price Targets for {TEST_TICKER}:")
pprint(price_targets)

In [ ]:
# 4d. Insider Sentiment
insider = finnhub.get_insider_sentiment(TEST_TICKER)
print(f"Insider Sentiment for {TEST_TICKER}:")
pprint(insider)

In [ ]:
# 4e. Company Peers
peers = finnhub.get_peers(TEST_TICKER)
print(f"Peers of {TEST_TICKER}: {peers}")

In [ ]:
# 4f. Full Sentiment Bundle (combines all above)
full_sentiment = finnhub.get_full_sentiment_data(TEST_TICKER)
print(f"Full Finnhub Data for {TEST_TICKER}:")
print(json.dumps(full_sentiment, indent=2, default=str))

---
## 5. Alpha Vantage — AI-Powered News Sentiment

**API key required:** `ALPHA_VANTAGE_API_KEY`  
**Rate limit:** 25 requests/day (free tier) — use sparingly!

In [ ]:
from src.agents.market_data.alpha_vantage_client import AlphaVantageClient

av = AlphaVantageClient()
print(f"Alpha Vantage client initialised (base URL: {av.base_url})")

In [ ]:
# 5a. Ticker-specific news sentiment (costs 1 of your 25 daily requests)
ticker_news = av.get_market_news_sentiment(tickers=TEST_TICKER, limit=10)

if "error" not in ticker_news:
    print(f"Alpha Vantage Sentiment for {TEST_TICKER}:")
    print(f"  Total articles:    {ticker_news['total_articles']}")
    print(f"  Average sentiment: {ticker_news['average_sentiment']:.4f}")
    print(f"  Bullish articles:  {ticker_news['bullish_articles']}")
    print(f"  Bearish articles:  {ticker_news['bearish_articles']}")
    print(f"  Neutral articles:  {ticker_news['neutral_articles']}")
    print()
    print("Top 5 articles:")
    for i, art in enumerate(ticker_news.get('articles', [])[:5], 1):
        score = art['overall_sentiment_score']
        label = art['overall_sentiment_label']
        print(f"  {i}. [{label:12s} {score:+.3f}] {art['title'][:80]}")
        print(f"     Source: {art['source']}")
else:
    print(f"Error: {ticker_news}")

In [ ]:
# 5b. Broad market sentiment (costs 1 of your 25 daily requests)
broad = av.get_broad_market_sentiment()

if "error" not in broad:
    print("Broad Market Sentiment (Alpha Vantage):")
    print(f"  Total articles:    {broad['total_articles']}")
    print(f"  Average sentiment: {broad['average_sentiment']:.4f}")
    print(f"  Bullish:           {broad['bullish_articles']}")
    print(f"  Bearish:           {broad['bearish_articles']}")
    print(f"  Neutral:           {broad['neutral_articles']}")
    
    # Show topic breakdown
    topic_counts = {}
    for art in broad.get('articles', []):
        for t in art.get('topics', []):
            topic_counts[t] = topic_counts.get(t, 0) + 1
    print(f"\n  Topic distribution:")
    for topic, count in sorted(topic_counts.items(), key=lambda x: -x[1])[:10]:
        print(f"    {topic:30s}: {count}")
else:
    print(f"Error: {broad}")

---
## 6. Trading 212 — Account, Portfolio, Instruments

**API keys required:** `T212_API_KEY`, `T212_API_SECRET`  
**Mode:** Practice/Demo (`demo.trading212.com`)

In [ ]:
from src.agents.execution.t212_client import T212Client

t212 = T212Client()
print(f"T212 client initialised (base URL: {t212.base_url})")

In [ ]:
# 6a. Account Cash Balance
try:
    cash = t212.get_cash()
    print("Account Cash:")
    pprint(cash)
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# 6b. Account Info
try:
    info = t212.get_account_info()
    print("Account Info:")
    pprint(info)
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# 6c. Current Portfolio (open positions)
try:
    portfolio = t212.get_portfolio()
    if portfolio:
        print(f"Open Positions ({len(portfolio)}):")
        for pos in portfolio:
            ticker = pos.get('ticker', 'N/A')
            qty = pos.get('quantity', 0)
            avg_price = pos.get('averagePrice', 0)
            current = pos.get('currentPrice', 0)
            pnl = pos.get('ppl', 0)
            print(f"  {ticker:10s}  qty={qty:>8.2f}  avg={avg_price:>8.2f}  current={current:>8.2f}  P&L={pnl:>+8.2f}")
    else:
        print("No open positions (empty portfolio)")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# 6d. Available Instruments (first 20)
try:
    instruments = t212.get_instruments()
    print(f"Total instruments available: {len(instruments)}")
    print(f"\nFirst 20:")
    for inst in instruments[:20]:
        ticker = inst.get('ticker', 'N/A')
        name = inst.get('name', 'N/A')[:40]
        currency = inst.get('currencyCode', '?')
        print(f"  {ticker:15s} {currency:4s} {name}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# 6e. Pending Orders
try:
    pending = t212.get_pending_orders()
    print(f"Pending Orders ({len(pending)}):")
    for order in pending:
        pprint(order)
    if not pending:
        print("  (none)")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# 6f. Order History (last 10)
try:
    history = t212.get_order_history(limit=10)
    print("Recent Order History:")
    pprint(history)
except Exception as e:
    print(f"Error: {e}")

---
## 7. Sub-Strategies — Momentum, Mean Reversion, Factor

These are pure Python, no LLM calls. They score stocks based on technical indicators and fundamentals.

In [ ]:
from src.agents.strategy.momentum import evaluate_momentum, MomentumSignal
from src.agents.strategy.mean_reversion import evaluate_mean_reversion, MeanReversionSignal
from src.agents.strategy.factor import calculate_factor_score, FactorScore, rank_by_factor

print("Strategy modules loaded")

In [ ]:
# 7a. Momentum Strategy
mom_signal = evaluate_momentum(
    ticker=TEST_TICKER,
    indicators=indicators,
    relative_strength=rs,
    current_holding=False,  # Change to True to see sell signals for held positions
)

print(f"Momentum Signal for {TEST_TICKER}:")
print(f"  Action:    {mom_signal.action}")
print(f"  Score:     {mom_signal.score:.0f}/100")
print(f"  Reasoning: {mom_signal.reasoning}")

In [ ]:
# 7b. Mean Reversion Strategy
mr_signal = evaluate_mean_reversion(
    ticker=TEST_TICKER,
    indicators=indicators,
    fundamentals=fundamentals,
    sector_avg_pe=25.0,  # Approximate sector average
    current_holding=False,
)

print(f"Mean Reversion Signal for {TEST_TICKER}:")
print(f"  Action:    {mr_signal.action}")
print(f"  Score:     {mr_signal.score:.0f}/100")
print(f"  Reasoning: {mr_signal.reasoning}")

In [ ]:
# 7c. Factor Strategy
# Calculate 6-month return for factor scoring
six_mo_return = None
if len(df) >= 126:
    six_mo_return = float(df['Close'].iloc[-1] / df['Close'].iloc[-126] - 1)

factor = calculate_factor_score(
    ticker=TEST_TICKER,
    fundamentals=fundamentals,
    indicators=indicators,
    relative_strength=rs,
    six_month_return=six_mo_return,
)

print(f"Factor Score for {TEST_TICKER}:")
print(f"  Composite: {factor.composite_score:.1f}/100")
print(f"  Value:     {factor.value_score:.1f}/100  (30% weight)")
print(f"  Quality:   {factor.quality_score:.1f}/100  (30% weight)")
print(f"  Momentum:  {factor.momentum_score:.1f}/100  (40% weight)")
print(f"  Reasoning: {factor.reasoning}")
print(f"  Components: {factor.components}")

In [ ]:
# 7d. Run all 3 strategies on multiple tickers to compare
COMPARE_TICKERS = ["AAPL", "MSFT", "GOOGL", "TSLA", "NVDA", "META", "AMZN", "JPM"]

print(f"{'Ticker':8s} | {'Mom Act':8s} {'Mom Score':10s} | {'MR Act':8s} {'MR Score':10s} | {'Factor':8s} {'V':5s} {'Q':5s} {'M':5s}")
print("-" * 95)

all_factors = []
for t in COMPARE_TICKERS:
    try:
        t_df = yf.download(t, period="1y", interval="1d", progress=False)
        if isinstance(t_df.columns, pd.MultiIndex):
            t_df.columns = t_df.columns.get_level_values(0)
        if t_df.empty or len(t_df) < 200:
            print(f"{t:8s} | Insufficient data")
            continue

        t_ind = calculate_indicators(t_df)
        t_fund = get_fundamentals(t)
        t_rs = calculate_relative_strength(t_df, benchmark_df)
        t_6mo = float(t_df['Close'].iloc[-1] / t_df['Close'].iloc[-126] - 1) if len(t_df) >= 126 else None

        t_mom = evaluate_momentum(t, t_ind, t_rs)
        t_mr = evaluate_mean_reversion(t, t_ind, t_fund)
        t_fac = calculate_factor_score(t, t_fund, t_ind, t_rs, t_6mo)
        all_factors.append(t_fac)

        print(f"{t:8s} | {t_mom.action:8s} {t_mom.score:8.0f}   | {t_mr.action:8s} {t_mr.score:8.0f}   | {t_fac.composite_score:6.1f}  {t_fac.value_score:4.1f} {t_fac.quality_score:4.1f} {t_fac.momentum_score:4.1f}")
    except Exception as e:
        print(f"{t:8s} | Error: {e}")

# Show factor ranking
if all_factors:
    print(f"\nFactor Ranking (top {len(all_factors)}):")
    for i, f in enumerate(rank_by_factor(all_factors, top_n=len(all_factors)), 1):
        print(f"  {i}. {f.ticker:8s} composite={f.composite_score:.1f}")

---
## 8. Strategy Engine — Claude Synthesis

This is the main LLM call. Claude takes all sub-strategy results + sentiment data and produces final trading decisions.

**API key required:** `ANTHROPIC_API_KEY`  
**Model:** `claude-sonnet-4-5-20250929`  
**Cost:** ~£0.01-0.03 per call

In [ ]:
# 8a. First, see exactly what prompt Claude receives
from src.agents.strategy.prompts import STRATEGY_SYSTEM_PROMPT, build_strategy_prompt

print("=== SYSTEM PROMPT ===")
print(STRATEGY_SYSTEM_PROMPT)
print()
print("=" * 60)

In [ ]:
# 8b. Build the exact user prompt that would be sent to Claude
# (using the data we already collected above)

sample_prompt = build_strategy_prompt(
    portfolio_state="Cash: £10,000 (100%). No open positions.",
    market_regime=f"Regime: {macro.get('market_regime', 'UNKNOWN')} | VIX: {macro.get('vix', 'N/A')}",
    momentum_proposals=f"- {TEST_TICKER}: {mom_signal.action} (score: {mom_signal.score:.0f}) — {mom_signal.reasoning}",
    mean_reversion_proposals=f"- {TEST_TICKER}: {mr_signal.action} (score: {mr_signal.score:.0f}) — {mr_signal.reasoning}",
    factor_proposals=f"- {TEST_TICKER}: composite={factor.composite_score:.0f} (V={factor.value_score:.0f} Q={factor.quality_score:.0f} M={factor.momentum_score:.0f}) — {factor.reasoning}",
    finnhub_sentiment=json.dumps(full_sentiment, indent=2, default=str)[:1000],
    alpha_vantage_sentiment=f"Avg sentiment: {ticker_news.get('average_sentiment', 'N/A')}" if 'error' not in ticker_news else "N/A",
    system_state="ACTIVE",
    vix=macro.get("vix"),
    cash_pct=100.0,
    max_position_pct=12.0,
    num_positions=0,
    max_positions=15,
    momentum_weight=0.35,
    mean_reversion_weight=0.30,
    factor_weight=0.35,
)

print("=== USER PROMPT (what Claude receives) ===")
print(sample_prompt[:3000])
if len(sample_prompt) > 3000:
    print(f"\n... [{len(sample_prompt)} total chars]")

In [ ]:
# 8c. Actually call Claude and see the response
# WARNING: This costs real money (~£0.01-0.03)
import anthropic

client = anthropic.Anthropic(api_key=settings.anthropic_api_key)

response = client.messages.create(
    model=settings.strategy_model,
    max_tokens=4096,
    system=STRATEGY_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": sample_prompt}],
)

# Show usage
print(f"Model:         {settings.strategy_model}")
print(f"Input tokens:  {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")

from src.utils.cost_tracker import calculate_cost
cost = calculate_cost("anthropic", response.usage.input_tokens, response.usage.output_tokens)
print(f"Cost:          £{cost:.4f}")
print()

# Parse and display Claude's decision
raw = response.content[0].text
print("=== RAW RESPONSE ===")
print(raw[:2000])
print()

# Parse JSON
try:
    content = raw
    if "```json" in content:
        content = content.split("```json")[1].split("```")[0]
    elif "```" in content:
        content = content.split("```")[1].split("```")[0]
    decisions = json.loads(content)

    print("=== PARSED DECISIONS ===")
    print(f"Market Assessment: {decisions.get('market_assessment', 'N/A')}")
    print(f"Portfolio Commentary: {decisions.get('portfolio_commentary', 'N/A')}")
    print()
    for d in decisions.get('decisions', []):
        print(f"  {d['ticker']:8s} {d['action']:6s} alloc={d.get('target_allocation_pct', 0):.1f}%  "
              f"conviction={d.get('conviction', 0)}  strategy={d.get('primary_strategy', '?')}")
        print(f"           Reasoning: {d.get('reasoning', 'N/A')[:100]}")
        print()
except json.JSONDecodeError as e:
    print(f"Failed to parse JSON: {e}")

---
## 9. Moderation Panel — GPT-4o & Gemini Reviews

The Investment Committee: 3 LLMs review each trade proposal.

- **GPT-4o** (Skeptical Analyst): Challenges assumptions, flags risks  
- **Gemini 2.0 Flash** (Risk Assessor): Scores growth/risk/confidence  
- **Consensus**: 3 AGREE = APPROVED, 2 AGREE = CAUTION, 2 DISAGREE = BLOCKED

In [ ]:
# Build a sample trade proposal (using Claude's decision if available, or a manual one)
sample_trade = {
    "ticker": TEST_TICKER,
    "action": "BUY",
    "target_allocation_pct": 5.0,
    "conviction": 75,
    "primary_strategy": "momentum",
    "reasoning": f"Strong momentum with RSI at {indicators.get('rsi_14', 0):.0f}, above 50-day MA, good relative strength vs S&P 500.",
    "growth_potential": "MEDIUM",
    "risk_level": "MEDIUM",
    "catalysts": ["Product launches", "AI integration"],
    "risks": ["Valuation stretch", "Macro headwinds"],
    "stop_loss_pct": -8.0,
    "upside_target_pct": 15.0,
}

# Try to use Claude's actual decision if we have one
try:
    if decisions and decisions.get('decisions'):
        sample_trade = decisions['decisions'][0]
        print(f"Using Claude's actual decision for {sample_trade['ticker']}")
except NameError:
    print(f"Using manual sample trade for {TEST_TICKER}")

portfolio_context = "Cash: £10,000 (100%). No open positions. Fresh portfolio."
sentiment_str = json.dumps(full_sentiment, indent=2, default=str)[:1500]

print(f"\nTrade proposal:")
pprint(sample_trade)

In [ ]:
# 9a. GPT-4o Skeptical Review
# Cost: ~£0.005-0.01
from src.agents.moderation import openai_mod

print("=== GPT-4o System Prompt ===")
print(openai_mod.SYSTEM_PROMPT)
print()

gpt4o_result = openai_mod.review_trade(
    trade_proposal=sample_trade,
    portfolio_context=portfolio_context,
    sentiment_data=sentiment_str,
)

print("=== GPT-4o Verdict ===")
print(f"  Verdict:    {gpt4o_result.get('verdict')}")
print(f"  Reasoning:  {gpt4o_result.get('reasoning')}")
print(f"  Risk flags: {gpt4o_result.get('risk_flags', [])}")
print(f"  Mods:       {gpt4o_result.get('modifications')}")
print(f"  Available:  {gpt4o_result.get('available')}")

In [ ]:
# 9b. Gemini Risk Assessment
# Cost: ~£0.001
from src.agents.moderation import gemini_mod

print("=== Gemini System Prompt ===")
print(gemini_mod.SYSTEM_PROMPT)
print()

gemini_result = gemini_mod.review_trade(
    trade_proposal=sample_trade,
    portfolio_context=portfolio_context,
    sentiment_data=sentiment_str,
)

print("=== Gemini Verdict ===")
print(f"  Verdict:        {gemini_result.get('verdict')}")
print(f"  Growth score:   {gemini_result.get('growth_score')}/10")
print(f"  Risk score:     {gemini_result.get('risk_score')}/10")
print(f"  Confidence:     {gemini_result.get('confidence_score')}/10")
print(f"  Assessment:     {gemini_result.get('assessment')}")
print(f"  High risk flag: {gemini_result.get('high_risk_flag')}")
print(f"  Modifications:  {gemini_result.get('modifications')}")
print(f"  Available:      {gemini_result.get('available')}")

In [ ]:
# 9c. Run the full Moderation Panel to see consensus logic
from src.agents.moderation.panel import ModerationPanel

panel = ModerationPanel()

mod_result = panel.review_trade(
    trade_proposal=sample_trade,
    portfolio_context=portfolio_context,
    sentiment_data=sentiment_str,
    conviction=sample_trade.get("conviction", 75),
    cycle_id="notebook_test",
)

print("=== MODERATION PANEL CONSENSUS ===")
print(f"  Ticker:        {mod_result.ticker}")
print(f"  CONSENSUS:     {mod_result.consensus}")
print(f"  Strategy:      {mod_result.strategy_verdict}")
print(f"  GPT-4o:        {mod_result.gpt4o_verdict.get('verdict') if mod_result.gpt4o_verdict else 'SKIP'}")
print(f"  Gemini:        {mod_result.gemini_verdict.get('verdict') if mod_result.gemini_verdict else 'SKIP'}")
print(f"  Moderators:    {mod_result.moderators_available}/2")
print(f"  Caution flag:  {mod_result.caution_flag}")

---
## 10. Risk Manager — Hard Rule Checks

Pure Python, no LLM calls. These rules have **VETO power** and can never be overridden by any LLM.

In [ ]:
from src.agents.risk.risk_manager import RiskManager

risk_mgr = RiskManager()

# Print all risk limits from settings
print("Risk Manager Configuration:")
print(f"  Max single stock:    {settings.max_single_stock_pct}%")
print(f"  Max sector:          {settings.max_sector_pct}%")
print(f"  Max correlation:     {settings.max_correlation}")
print(f"  Cautious drawdown:   {settings.cautious_drawdown_pct}%")
print(f"  Halt drawdown:       {settings.halt_drawdown_pct}%")
print(f"  Daily loss halt:     {settings.daily_loss_halt_pct}%")
print(f"  Cash floor:          {settings.cash_floor_pct}%")
print(f"  VIX high:            {settings.vix_high}")
print(f"  VIX extreme:         {settings.vix_extreme}")
print(f"  Min positions:       {settings.min_positions}")

In [ ]:
# 10a. Test individual rules

# Max single stock
r1 = risk_mgr.check_max_single_stock("AAPL", 15.0, {"MSFT": 8.0})
print(f"Max single stock (15% AAPL): {r1.passed} — {r1.message}")
if r1.adjusted_allocation:
    print(f"  Adjusted to: {r1.adjusted_allocation}%")

r2 = risk_mgr.check_max_single_stock("AAPL", 8.0, {"MSFT": 8.0})
print(f"Max single stock (8% AAPL):  {r2.passed} — {r2.message}")
print()

In [ ]:
# Max sector concentration
r3 = risk_mgr.check_max_sector("AAPL", "Technology", 10.0, {"Technology": 25.0})
print(f"Sector limit (Tech at 25% + 10%): {r3.passed} — {r3.message}")

r4 = risk_mgr.check_max_sector("AAPL", "Technology", 5.0, {"Technology": 20.0})
print(f"Sector limit (Tech at 20% + 5%):  {r4.passed} — {r4.message}")
print()

In [ ]:
# VIX-based limits
r5 = risk_mgr.check_vix_limit(vix=15.0, proposed_pct=10.0)
print(f"VIX=15 (calm), 10% position:    {r5.passed} — {r5.message}")

r6 = risk_mgr.check_vix_limit(vix=28.0, proposed_pct=10.0)
print(f"VIX=28 (high), 10% position:    {r6.passed} — {r6.message}")

r7 = risk_mgr.check_vix_limit(vix=40.0, proposed_pct=10.0)
print(f"VIX=40 (extreme), 10% position: {r7.passed} — {r7.message}")
print()

In [ ]:
# Cash floor check
r8 = risk_mgr.check_cash_floor(cash_pct=50.0, trade_pct=10.0)
print(f"Cash 50%, trade 10%: {r8.passed} — {r8.message}")

r9 = risk_mgr.check_cash_floor(cash_pct=15.0, trade_pct=8.0)
print(f"Cash 15%, trade 8%:  {r9.passed} — {r9.message}")
print()

In [ ]:
# Drawdown state check
r10 = risk_mgr.check_drawdown(current_value=9800, peak_value=10000)
print(f"2% drawdown:  {r10.passed} — {r10.message}")
print(f"  State: {risk_mgr.get_drawdown_state(9800, 10000)}")

r11 = risk_mgr.check_drawdown(current_value=9200, peak_value=10000)
print(f"8% drawdown:  {r11.passed} — {r11.message}")
print(f"  State: {risk_mgr.get_drawdown_state(9200, 10000)}")

r12 = risk_mgr.check_drawdown(current_value=8000, peak_value=10000)
print(f"20% drawdown: {r12.passed} — {r12.message}")
print(f"  State: {risk_mgr.get_drawdown_state(8000, 10000)}")
print()

In [ ]:
# 10b. Full trade evaluation (all rules together)
verdict = risk_mgr.evaluate_trade(
    ticker=TEST_TICKER,
    action="BUY",
    proposed_allocation_pct=8.0,
    sector=fundamentals.get("sector", "Technology"),
    current_portfolio={"MSFT": 8.0, "GOOGL": 7.0, "NVDA": 6.0, "JPM": 5.0, "JNJ": 4.0},
    sector_allocations={"Technology": 21.0, "Financials": 5.0, "Healthcare": 4.0},
    portfolio_returns={},  # Empty — correlation check will be skipped
    current_value=9800,
    peak_value=10000,
    cash_pct=65.0,
    vix=macro.get("vix"),
    daily_pnl_pct=-0.5,
    daily_loss_halt_until=None,
    num_positions=5,
    system_state="ACTIVE",
    cycle_id="notebook_test",
)

print(f"=== RISK VERDICT for {TEST_TICKER} BUY 8% ===")
print(f"  Verdict:              {verdict.verdict}")
print(f"  Proposed allocation:  {verdict.proposed_allocation_pct:.1f}%")
print(f"  Adjusted allocation:  {verdict.adjusted_allocation_pct}%" if verdict.adjusted_allocation_pct else "  Adjusted allocation:  N/A")
print(f"  Rules checked:        {verdict.rules_checked}")
print(f"  Triggered rules:      {verdict.triggered_rules}")
print(f"  Reasoning:            {verdict.reasoning}")

---
## Summary

| Section | Data Source | API Key | Cost | Status |
|---------|-----------|---------|------|--------|
| 2 | yfinance | None | Free | Check cells above |
| 3 | Technical Indicators (ta) | None | Free | Check cells above |
| 4 | Finnhub | `FINNHUB_API_KEY` | Free tier (60 req/min) | Check cells above |
| 5 | Alpha Vantage | `ALPHA_VANTAGE_API_KEY` | Free tier (25 req/day) | Check cells above |
| 6 | Trading 212 | `T212_API_KEY` + `T212_API_SECRET` | Free (Demo) | Check cells above |
| 7 | Sub-strategies | None | Free (local) | Check cells above |
| 8 | Claude (Anthropic) | `ANTHROPIC_API_KEY` | ~£0.01-0.03/call | Check cells above |
| 9 | GPT-4o + Gemini | `OPENAI_API_KEY` + `GOOGLE_AI_API_KEY` | ~£0.005 + £0.001/call | Check cells above |
| 10 | Risk Manager | None | Free (local) | Check cells above |